# 02 - Preprocessing & Spatial Split

Loads the per-file feature CSVs produced by `01_feature_extraction.ipynb`, combines them into a **canonical, file based train/test split**  , handles NaN points with full visibility, maps raw labels to class indices, fits normalization and writes the final `train.csv`/`test.csv` split artifacts that every later notebook (uncertainty fit, model training, PAvPU) should read from.

In [ ]:
import sys
from pathlib import Path

#sys.path.insert(0, str(Path.cwd().parent if (Path.cwd() / "notebooks").exists() else Path.cwd()))
#sys.path.insert(0, str(Path.cwd().parent if (Path.cwd() / "notebooks").exists() else Path.cwd()))
current_notebook_dir = Path.cwd()

if current_notebook_dir.name == "notebooks":
    project_root = current_notebook_dir.parent
else:
    project_root = current_notebook_dir

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd

from src.utils import set_global_seed, load_config, get_logger
from data.paths import get_dataset_config, get_split_file
from src.spatial_split import load_split
from src.classes import get_class_registry
from src.normalization import fit_scaler, apply_scaler
from src.features import DEFAULT_FEATURE_NAMES

logger = get_logger("02_preprocessing_and_split")

In [ ]:
DATASET = "ahn3"

config = load_config("default")
set_global_seed(config["experiment"]["seed"])
dataset_cfg = get_dataset_config(DATASET)

## 1. Load the canonical file-based split

`load_split()` reads data splits and concatenates the declared train / test files into two separate DataFrames. This will refuse to run if the dataset is misconfigured

In [ ]:
train_df, test_df = load_split(DATASET)

print(f"train: {len(train_df):,} points from {sorted(train_df['__source_file__'].unique())}")
print(f"test:  {len(test_df):,} points from {sorted(test_df['__source_file__'].unique())}")

#Check if train and test files overlaps. They sjhouldnt
overlap = set(train_df["__source_file__"].unique()) & set(test_df["__source_file__"].unique())
assert not overlap, f"Train/test file overlap detected {overlap}"
print("OK: no file overlap between train and test.")

## 2. Handle NaN points

Points flagged `__nan_feature__=True` in has insufficient neighbors within the feature extraction radius. We report the breakdown by class **before** dropping them, separately for train and test, so we have visibility into whether this disproportionately removes any particular
class

In [ ]:
def report_and_drop_nan(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    n_nan = int(df["__nan_feature__"].sum())
    if n_nan > 0:
        pct = 100 * n_nan / len(df)
        logger.warning(f"[{split_name}] dropping {n_nan:,} / {len(df):,} ({pct:.2f}%) NaN-flagged points")
        print(f"[{split_name}] class distribution of DROPPED points:")
        print(df.loc[df["__nan_feature__"], "label"].value_counts())
    return df.loc[~df["__nan_feature__"]].reset_index(drop=True)

train_df = report_and_drop_nan(train_df, "train")
test_df = report_and_drop_nan(test_df, "test")

## 3. Map raw labels to class indices

In [ ]:
class_registry = get_class_registry(DATASET)
print(f"num_classes = {class_registry.num_classes}")
print(class_registry.class_names_in_order())

train_df["class_idx"] = class_registry.map_labels(train_df["label"].values)
test_df["class_idx"] = class_registry.map_labels(test_df["label"].values)

## 4. Normalize : fit on TRAIN only, apply to both

`fit_scaler` is called on `train_df` alone to prevent the dataleakage. x, y, z are included in the scaled columns and locally re-centered per source file first, so absolute spatial position is removed as a learnable signal rather than being fed in raw and unscaled alongside scaled features

In [ ]:
feature_columns = ["x", "y", "z"] + DEFAULT_FEATURE_NAMES

scaler = fit_scaler(train_df, feature_columns, config)   # FIT ON TRAIN ONLY
train_scaled = apply_scaler(train_df, scaler, feature_columns)
test_scaled = apply_scaler(test_df, scaler, feature_columns)   # transform only, never re-fit

print("train x range after scaling:", train_scaled["x"].min(), "-", train_scaled["x"].max())
print("test  x range after scaling:", test_scaled["x"].min(), "-", test_scaled["x"].max())

## 5. Save the canonical split artifacts

In [ ]:
train_out = get_split_file(DATASET, "train")
test_out = get_split_file(DATASET, "test")

train_scaled.to_csv(train_out, index=False)
test_scaled.to_csv(test_out, index=False)

print(f"Saved train split -> {train_out} ({len(train_scaled):,} rows)")
print(f"Saved test split  -> {test_out} ({len(test_scaled):,} rows)")